# CISM Tutorial 02: FANMOD+ And CISM Initialization

This notebook starts from the output of tutorial 01.

You should already have:

- `network_dataset_root_path`
- `dataset_folder`
- a dataset directory containing `Patient_<patient_id>_FOV<fov>.txt` files
- a `patient_class.csv` in the same dataset directory

The goal of this notebook is to:

1. point `CISM` to the FANMOD+ binary bundled in this repository
2. create tutorial-local `fanmod_output` and `fanmod_cache` folders
3. initialize `CISM`
4. load the prepared dataset with the tutorial defaults

## Notes About FANMOD+ Binaries

For now this tutorial uses the FANMOD+ binaries already stored under the `cism` folder.

That keeps the tutorial self-contained and avoids extra setup during onboarding.
If the repo later grows a dedicated top-level `binaries/` or `external_tools/` folder, the same notebook structure can be updated easily.

In [ ]:
from pathlib import Path
import platform

from cism import CISM, validate_network_dataset_directory


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "tutorials" else Path.cwd().resolve()
TUTORIALS_DIR = PROJECT_ROOT / "tutorials"
TUTORIAL_RUNTIME_DIR = TUTORIALS_DIR / "runtime"
FANMOD_OUTPUT_ROOT = TUTORIAL_RUNTIME_DIR / "fanmod_output"
FANMOD_CACHE_ROOT = TUTORIAL_RUNTIME_DIR / "fanmod_cache"

FANMOD_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FANMOD_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Tutorial runtime directory: {TUTORIAL_RUNTIME_DIR}")
print(f"fanmod_output_root_path: {FANMOD_OUTPUT_ROOT}")
print(f"fanmod_cache_root_path: {FANMOD_CACHE_ROOT}")


## Paste The Handoff Values From Tutorial 01

In tutorial 01, the notebook printed the exact values to use here.

- `network_dataset_root_path` should be the parent directory that contains your dataset folder.
- `dataset_folder` should be the dataset subfolder name itself.

Example:

- `network_dataset_root_path = /.../data`
- `dataset_folder = example_dataset`

In [ ]:
# Replace these with the values printed at the end of tutorial 01.
network_dataset_root_path = str(PROJECT_ROOT / "data")
dataset_folder = "example_dataset"

dataset_dir = Path(network_dataset_root_path) / dataset_folder
print(f"network_dataset_root_path = {network_dataset_root_path}")
print(f"dataset_folder = {dataset_folder}")
print(f"dataset_dir = {dataset_dir}")


In [ ]:
# Validate that tutorial 01 really produced a usable dataset folder.
txt_files = validate_network_dataset_directory(dataset_dir, require_patient_class=True)
print(f"Validated {len(txt_files)} txt graph files in {dataset_dir}")


## Resolve The FANMOD+ Binary

This repository currently ships FANMOD+ binaries under `cism/FANMOD_binaries/`.

The `CISM` object expects:

- `fanmod_path`: directory containing the binary
- `fanmod_exe`: the binary filename

In [ ]:
fanmod_path = PROJECT_ROOT / "cism" / "FANMOD_binaries"

if platform.system().lower().startswith("win"):
    fanmod_exe = "LocalFANMOD - Windows"
else:
    fanmod_exe = "LocalFANMOD"

fanmod_binary_path = fanmod_path / fanmod_exe
print(f"fanmod_path = {fanmod_path}")
print(f"fanmod_exe = {fanmod_exe}")
print(f"fanmod binary exists: {fanmod_binary_path.exists()}")


## Initialize `CISM`

This tutorial uses the requested defaults:

- `motif_size = 3`
- `iterations = 1000`
- `force_run_fanmod = False`
- `force_parse = False`
- `n_jobs = 12`

In [ ]:
cism = CISM(
    fanmod_exe=fanmod_exe,
    fanmod_path=str(fanmod_path),
    network_dataset_root_path=network_dataset_root_path,
    fanmod_output_root_path=str(FANMOD_OUTPUT_ROOT),
    fanmod_cache_root_path=str(FANMOD_CACHE_ROOT),
    motif_size=3,
    iterations=1000,
)

cism


## Load The Prepared Dataset

This step runs FANMOD+ as needed and parses its outputs into `cism.motifs_dataset`.

For this tutorial, use:

- `force_run_fanmod=False`
- `force_parse=False`
- `n_jobs=12`

In [ ]:
# Replace dataset_type and dataset_name with the metadata you want to carry into analysis.
dataset_type = "Disease"
dataset_name = "Example"

cism.add_dataset(
    dataset_folder=dataset_folder,
    dataset_type=dataset_type,
    dataset_name=dataset_name,
    force_run_fanmod=False,
    force_parse=False,
    n_jobs=12,
    require_patient_class=True,
)


In [ ]:
motif_df = cism.motif_dataset()
print(f"Loaded motif rows: {0 if motif_df is None else len(motif_df)}")
display(motif_df.head() if motif_df is not None else None)


In [ ]:
fanmod_dataset_output_dir = FANMOD_OUTPUT_ROOT / dataset_folder
fanmod_dataset_cache_dir = FANMOD_CACHE_ROOT / dataset_folder

print("Initialization summary:")
print(f"network_dataset_root_path = {network_dataset_root_path}")
print(f"dataset_folder = {dataset_folder}")
print(f"fanmod_path = {fanmod_path}")
print(f"fanmod_exe = {fanmod_exe}")
print(f"fanmod_output_root_path = {FANMOD_OUTPUT_ROOT}")
print(f"fanmod_cache_root_path = {FANMOD_CACHE_ROOT}")
print(f"dataset-specific FANMOD output dir = {fanmod_dataset_output_dir}")
print(f"dataset-specific FANMOD cache dir = {fanmod_dataset_cache_dir}")


## You Are Done When...

Before moving to analysis, confirm that:

- your tutorial 01 dataset folder validated successfully
- the FANMOD+ binary path resolves correctly
- `CISM(...)` was initialized with `motif_size=3` and `iterations=1000`
- `cism.add_dataset(...)` completed with:
  - `force_run_fanmod=False`
  - `force_parse=False`
  - `n_jobs=12`
- `cism.motif_dataset()` is populated

At that point, you are ready for the next notebook: **analysis**.